In [7]:
import pandas as pd
from pathlib import Path

DATA_FILE = "dev_agent_combined.csv"

def find_data_dir(start: Path, marker: str = "dataset/data") -> Path:
    for parent in [start, *start.parents]:
        candidate = parent / marker
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(f"Could not locate '{marker}' above {start}")

DATA_DIR = find_data_dir(Path.cwd().resolve())
DATA_PATH = DATA_DIR / DATA_FILE

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows from {DATA_PATH}")
df["group"].value_counts()


Loaded 12276 rows from G:\On the Naturalness of Agent-Generated Documentation\dataset\data\dev_agent_combined.csv


group
agent    6311
human    5965
Name: count, dtype: int64

In [8]:
# 100 random agent-generated documentation samples
SEED = 42

agent_samples = df[df["group"] == "agent"]
agent_samples = agent_samples[agent_samples["doc_text"].notna()]

agent_samples = agent_samples.sample(n=100, random_state=SEED)


for i, row in agent_samples.iterrows():
 #   print(f"--- [{i}] {row['repo']} PR#{row['pull_request']} | {row['function_name']} ---")
    print(row["doc_text"] if pd.notna(row["doc_text"]) else "(no docstring)")
    print()


Get current watchlist

Setup logging for the converter Clear existing handlers Set logging level Create formatter Console handler File handler if log_dir is provided

Handle delete action (either selected annotation or all annotations) Delete selected annotation Delete all annotations from current image

Log warning message

Đếm số từ

ObjectMapper 설정: ISO-8601 UTC 형식으로 통일 Frontend ↔ Backend 간 일관된 날짜 형식 보장 JavaTimeModule 등록 (java.time.* 타입 지원) Instant를 ISO-8601 UTC 형식으로 직렬화 예: "2024-01-15T10:30:00.123Z" TIMESTAMP를 숫자가 아닌 ISO-8601 문자열로 직렬화 null 값 처리 설정

pragma: no cover - hardware errors

Load model from JSON file Convert back to ProjectModel Convert file data back to FileModel objects

Wait for the transition to complete before hiding

Delete the selected polygon Remove from annotations array Remove edit handles Remove from canvas Clear active polygon Update displays Update button states

Updates an existing Pomodoro session.

Just verify that the profiler was created successfully The 

In [ ]:
# 100 random developer (human) documentation samples
dev_samples = df[df["group"] == "human"]
dev_samples = dev_samples[dev_samples["doc_text"].notna()]

dev_samples = dev_samples.sample(n=100, random_state=SEED)

for i, row in dev_samples.iterrows():
#    print(f"--- [{i}] {row['repo']} PR#{row['pull_request']} | {row['function_name']} ---")
    print(row["doc_text"] if pd.notna(row["doc_text"]) else "(no docstring)")
    print()


One round of ask-tell must have been run One round of ask-tell must have been run New position must have correct size Proposal can be changed too New proposal must have correct size

brute force the correct answer by testing every partition.

add 1-3 to cache (so enter inner loop) consume 4-5 already cached add 6-7 to cache (so enter inner loop) consume 8-10 already cached consume 1-3 already cached add 4-5 to cache (so go to outer loop) consume 1-7 already cached add 8-10 to cache (so go to outer loop) consume 1-10 (all item were already cached)

Fetches a C-style null terminated char[] from the specified buffer, returning a {@link ByteString} representation. @param buffer to fetch the string from @param offset index within the buffer to commence the null character search @return the located string

`str` `artistUrl` : Spotify Url of the artist whose tracks are to be retrieved returns a `List` containing Url's of each track of the artist. Check if the artist has more albums.

In case 

## What is `doc_code_overlap` actually made of?

`doc_code_overlap` is computed in [build.py:125](../../dataset/buildDataset/build.py) as

```python
doc_tokens = set(tokenize(doc_text))
code_tokens = set(tokenize(code_text_no_documentation))
overlap = len(doc_tokens & code_tokens) / len(doc_tokens)
```

where `tokenize` matches identifier-shaped words (`[A-Za-z_][A-Za-z0-9_]*`, lowercased) and
`code_text_no_documentation` is the function body with all comments/docstrings stripped by tree-sitter.
So it is the **fraction of distinct documentation word types that also appear somewhere in the code** —
it counts English stopwords that happen to be language keywords (`if`, `for`, `in`, `return`, `is`, `not`,
`a`) just as readily as identifiers echoed from the signature.

The cells below re-derive the metric from the `function` + `doc_text` columns (verified to reproduce the
stored values exactly) and print the individual overlapping tokens.

In [4]:
import re
from collections import Counter
from tree_sitter import Parser as _TSParser
from tree_sitter_language_pack import get_language as _ts_get_language

SEED = 42
N_SAMPLES = 50
MAX_DOC_CHARS = 500   # truncate long docstrings when printing

# ---- copied verbatim from dataset/buildDataset/build.py so the metric is reproduced exactly ----
_TS_LANGUAGE_MAP = {
    '.py': 'python',
    '.js': 'javascript', '.jsx': 'javascript',
    '.java': 'java',
    '.c': 'c', '.h': 'c',
    '.cpp': 'cpp', '.cc': 'cpp', '.hpp': 'cpp', '.cxx': 'cpp', '.hxx': 'cpp',
    '.cs': 'csharp',
    '.go': 'go',
    '.kt': 'kotlin', '.kts': 'kotlin',
    '.php': 'php',
    '.scala': 'scala',
    '.swift': 'swift',
}
_ts_parser_cache = {}

def _get_ts_parser(file_extension):
    lang_name = _TS_LANGUAGE_MAP.get(file_extension.lower())
    if lang_name is None:
        return None
    if lang_name not in _ts_parser_cache:
        _ts_parser_cache[lang_name] = _TSParser(_ts_get_language(lang_name))
    return _ts_parser_cache[lang_name]

def strip_comments(text, file_extension=None):
    ext = file_extension.lower() if file_extension else ""
    parser = _get_ts_parser(ext)
    if parser is None:
        return text

    parse_text = "<?php\n" + text if ext == ".php" else text
    parse_bytes = parse_text.encode("utf-8")
    tree = parser.parse(parse_bytes)
    remove_ranges = []

    def visit(node):
        if "comment" in node.type:
            remove_ranges.append((node.start_byte, node.end_byte))
            return
        if (ext == ".py" and node.type == "string"
                and node.parent is not None and node.parent.type in ("module", "block")):
            remove_ranges.append((node.start_byte, node.end_byte))
            return
        for child in node.children:
            visit(child)

    visit(tree.root_node)
    remove_ranges.sort()

    kept, cursor = bytearray(), 0
    for start, end in remove_ranges:
        kept += parse_bytes[cursor:start]
        cursor = end
    kept += parse_bytes[cursor:]

    result = kept.decode("utf-8", errors="replace")
    if ext == ".php":
        result = result[len("<?php\n"):]
    return "\n".join(line.rstrip() for line in result.splitlines() if line.strip())

def tokenize(text):
    if not text:
        return []
    return re.findall(r"[A-Za-z_][A-Za-z0-9_]*", str(text).lower())
# ---- end verbatim ----


def name_subtokens(name):
    """snake_case / camelCase sub-tokens of an identifier, used to flag signature echoes."""
    parts = []
    for t in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", str(name)):
        for p in re.split(r"_+", t):
            parts.extend(re.findall(r"[A-Z]+(?![a-z])|[A-Z][a-z0-9]*|[a-z0-9]+", p))
    return {p.lower() for p in parts if p}


def overlap_breakdown(row):
    """Recompute doc_code_overlap for one row and expose *which* tokens overlap."""
    ext = Path(str(row["file_path"])).suffix.lower()
    code_no_doc = strip_comments(str(row["function"]), ext)   # exactly what the metric compares against
    doc_tokens = set(tokenize(row["doc_text"]))
    code_tokens = set(tokenize(code_no_doc))
    shared = doc_tokens & code_tokens
    ratio = 0.0 if not doc_tokens or not code_tokens else len(shared) / len(doc_tokens)
    return {
        "ext": ext,
        "doc_tokens": doc_tokens,
        "code_tokens": code_tokens,
        "shared": shared,
        "doc_only": doc_tokens - code_tokens,
        "ratio": ratio,
        "in_func_name": shared & name_subtokens(row["function_name"]),
    }


documented = df[df["doc_code_overlap"].notna() & df["doc_text"].notna() & df["function"].notna()]
print(f"{len(documented)} documented functions "
      f"({(documented['group'] == 'agent').sum()} agent / {(documented['group'] == 'human').sum()} human)")

# sanity check: the recomputation must match the stored column
_check = documented.sample(n=200, random_state=0)
_diff = max(abs(round(overlap_breakdown(r)["ratio"], 4) - r["doc_code_overlap"]) for _, r in _check.iterrows())
print(f"max |recomputed - stored| over 200 random rows: {_diff:.6f}")

5216 documented functions (3264 agent / 1952 human)
max |recomputed - stored| over 200 random rows: 0.000050


In [5]:
# 50 random samples per group, showing which tokens the overlap consists of
def show_overlap_samples(group, n=N_SAMPLES, seed=SEED):
    sub = documented[documented["group"] == group].sample(n=n, random_state=seed)
    label = {"agent": "AGENT", "human": "DEVELOPER"}[group]
    print("#" * 100)
    print(f"# {n} random {label} samples - doc_code_overlap breakdown")
    print("#   SHARED   = doc tokens that also occur in the comment-stripped code (the numerator)")
    print("#   DOC-ONLY = doc tokens with no counterpart in the code")
    print("#   a trailing * marks a shared token that is also a sub-token of the function name")
    print("#" * 100)

    for i, row in sub.iterrows():
        b = overlap_breakdown(row)
        doc = " ".join(str(row["doc_text"]).split())
        if len(doc) > MAX_DOC_CHARS:
            doc = doc[:MAX_DOC_CHARS] + " ...[truncated]"

        print(f"\n--- [{i}] {row['repo']} PR#{row['pull_request']} | {row['function_name']} ({b['ext']}) "
              f"| {row['label'] if pd.notna(row['label']) else 'human'} ---")
        print(f"    overlap = {b['ratio']:.4f} (stored {row['doc_code_overlap']:.4f}) "
              f"= {len(b['shared'])} shared / {len(b['doc_tokens'])} distinct doc tokens")
        print(f"    DOC : {doc}")
        shared_str = ", ".join(t + ("*" if t in b["in_func_name"] else "")
                               for t in sorted(b["shared"])) or "(none)"
        print(f"    SHARED   ({len(b['shared'])}): {shared_str}")
        print(f"    DOC-ONLY ({len(b['doc_only'])}): {', '.join(sorted(b['doc_only'])) or '(none)'}")


show_overlap_samples("agent")
print("\n" * 2)
show_overlap_samples("human")

####################################################################################################
# 50 random AGENT samples - doc_code_overlap breakdown
#   SHARED   = doc tokens that also occur in the comment-stripped code (the numerator)
#   DOC-ONLY = doc tokens with no counterpart in the code
#   a trailing * marks a shared token that is also a sub-token of the function name
####################################################################################################

--- [2675] https://api.github.com/repos/Yushcheng777/a-stock-trend-strategy PR#4 | test_custom_parameters (.py) | Copilot ---
    overlap = 0.0000 (stored 0.0000) = 0 shared / 11 distinct doc tokens
    DOC : Test strategy with custom parameters. Check that custom parameters are set Test that strategy still works with custom parameters
    SHARED   (0): (none)
    DOC-ONLY (11): are, check, custom, parameters, set, still, strategy, test, that, with, works

--- [280] https://api.github.com/repos/kellylford/Im

In [6]:
# Corpus-wide: which tokens does the overlap actually consist of? (runs over all documented rows)
rows = []
for i, row in documented.iterrows():
    b = overlap_breakdown(row)
    rows.append({
        "idx": i, "group": row["group"], "shared": b["shared"],
        "n_doc": len(b["doc_tokens"]), "n_shared": len(b["shared"]),
        "n_in_name": len(b["in_func_name"]),
    })
ov = pd.DataFrame(rows)

print("=== Most frequent overlapping tokens (% of that group's documented functions) ===\n")
tops = {}
for g in ["agent", "human"]:
    counts = Counter(t for s in ov.loc[ov["group"] == g, "shared"] for t in s)
    n_docs = (ov["group"] == g).sum()
    tops[g] = [(t, k / n_docs) for t, k in counts.most_common(30)]

print(f"{'rank':>4}  {'AGENT token':<22}{'% of docs':>10}   {'DEVELOPER token':<22}{'% of docs':>10}")
print("-" * 78)
for r in range(30):
    a_t, a_p = tops["agent"][r]
    h_t, h_p = tops["human"][r]
    print(f"{r+1:>4}  {a_t:<22}{100*a_p:>9.1f}%   {h_t:<22}{100*h_p:>9.1f}%")

print("\n=== How much of the overlap is the function signature restated? ===")
for g in ["agent", "human"]:
    s = ov[ov["group"] == g]
    share = s["n_in_name"].sum() / s["n_shared"].sum()
    print(f"  {g:<6}: {100*share:.1f}% of all shared tokens are sub-tokens of the function name "
          f"| median {s['n_shared'].median():.0f} shared of {s['n_doc'].median():.0f} distinct doc tokens")

=== Most frequent overlapping tokens (% of that group's documented functions) ===

rank  AGENT token            % of docs   DEVELOPER token        % of docs
------------------------------------------------------------------------------
   1  if                         10.0%   if                         11.8%
   2  for                         7.7%   for                        11.5%
   3  test                        5.9%   in                          8.5%
   4  to                          5.2%   return                      7.6%
   5  in                          4.7%   is                          2.8%
   6  return                      4.4%   data                        2.8%
   7  with                        3.4%   a                           2.7%
   8  image                       3.1%   to                          2.6%
   9  get                         2.8%   none                        2.3%
  10  data                        2.3%   not                         2.2%
  11  file              